In [1]:
import numpy as np

In [2]:
# define the science bias formula and a function to calculate it

def sb_calculator(citations_y1, citations_y2, IF_y1, IF_y2):
    sb_index =  (IF_y1 + IF_y2 + 1)/(citations_y1 + citations_y2 + 1)
    return sb_index

In [21]:
# create fictitious citation data
# Parameters
mean1 = 4
mean2 = 6
N = 1000  # Number of data points

year_1_citations = np.round(np.random.exponential(scale=2, size=N) + np.random.uniform(0, mean1-1, size=N))
year_2_citations = np.round(np.random.exponential(scale=2, size=N) + np.random.uniform(0, mean2-1, size=N))

In [22]:
# simulate impact factor of the Journal of Middle Earth
year_1_IF = np.round(np.mean(year_1_citations),1)
year_2_IF = np.round(np.mean(year_2_citations),1)

In [23]:
year_1_IF

3.4

In [24]:
year_2_IF

4.6

In [7]:
# identify the high and low bounds of the citation distributions
year_1_lower_bound = np.percentile(year_1_citations, 10)
year_1_upper_bound = np.percentile(year_1_citations, 90)

year_2_lower_bound = np.percentile(year_2_citations, 10)
year_2_upper_bound = np.percentile(year_2_citations, 90)

In [8]:
# now let's simulate countries (but may be applied to any other entity)
# that consistently publish papers with citations above or below 
# a journal's 10% and 90% impact percentile
# and calculate their bias in science index

Gondor_indices = []
Mordor_indices = []

In [9]:
for c1,c2 in zip(year_1_citations[year_1_citations >= year_1_upper_bound],year_2_citations[year_2_citations >= year_2_upper_bound]):
    Gondor_indices.append(sb_calculator(c1,c2,year_1_IF,year_2_IF))

In [10]:
np.round(np.median(Gondor_indices),2)

0.53

In [11]:
for a,b in zip(year_1_citations[year_1_citations <= year_1_lower_bound],year_2_citations[year_2_citations <= year_2_lower_bound]):
    Mordor_indices.append(sb_calculator(a,b,year_1_IF,year_2_IF))

In [12]:
np.round(np.median(Mordor_indices),2)

2.38

In [13]:
# if instead, we have some entity whose papers are cited as often
# as the journal's impact factor then the bs index would be close to 1..
#  lower than 1.2 is okay
sb_calculator(4, 4, 4.9, 4.9)

1.2000000000000002

# Concerning the journal bias index, i can't recall exactly what was mentiond during our call.
# if you meant by looking at a journal to show that the sb index is equal to 1:
# it isn't :/ even applying Alexandre's fix (inverting numerator and denominator in sb formula)
# IF is calculated on citations of all papers in a journal, BS on citations of single paper, they have diff values

In [16]:
import pandas as pd
bsaf_data = pd.read_csv('/Users/mococomac/Downloads/BSAF_2012_bsIndices.csv')

In [17]:
bsaf_data

,title,IF y+1,IF y+2,citations y+1,citations y+2,Paper BS,average BS
0,A cytoarchitectonic and chemoarchitectonic ana...,4.567,5.618,4,12,0.657941,2.739647
1,A deconvolution method to improve automated 3D...,4.567,5.618,6,1,1.398125,NaN
2,Amygdalar connections in the lesser hedgehog t...,4.567,5.618,0,0,11.185000,NaN
3,Brain grey matter deficits in smokers: focus o...,4.567,5.618,2,6,1.242778,NaN
4,Brain structure and function in borderline per...,4.567,5.618,5,14,0.559250,NaN
5,Brainpeps: the blood-brain barrier peptide dat...,4.567,5.618,0,2,3.728333,NaN
6,Cellular markers of neuroinflammation and neur...,4.567,5.618,2,4,1.597857,NaN
7,Cellular signatures in the primary visual cort...,4.567,5.618,2,1,2.796250,NaN
8,Collateral projections from nucleus reuniens o...,4.567,5.618,5,3,1.242778,NaN
9,Comparison of the effects of acute and chronic...,4.567,5.618,10,4,0.745667,NaN


calculate the sb indices of each paper published in 2012 in BSAF

In [18]:
bsaf_sb_indices = []
for i in range(bsaf_data.shape[0]):
    bsaf_sb_indices.append(sb_calculator(bsaf_data.iloc[i]['citations y+1'],bsaf_data.iloc[i]['citations y+2'],bsaf_data.iloc[i]['IF y+1'],bsaf_data.iloc[i]['IF y+2']))
    

In [53]:
print('median SB index for BSAF in 2012 is {}'.format(np.median(bsaf_sb_indices)))

median SB index for BSAF in 2012 is 1.8641666666666665


a bit high..

In [50]:
print('{}% of papers published in 2012 in BSAF had total citations \n after two years lower than the combined IFs'.format(np.round((np.sum(bsaf_data['citations y+1'] + bsaf_data['citations y+2'] < (4.567 + 5.618)) / bsaf_data.shape[0]) * 100,2)))

81.82% of papers published in 2012 in BSAF had total citations 
 after two years lower than the combined IFs


makes sense then! most weren't far off and that is why it still still
 reasonably below 2.